# 📊 EDA - Olist E-Commerce Reviews
## Análise Exploratória para Voice of Customer Intelligence com RAG

**Objetivo:** Entender a estrutura e qualidade dos dados de avaliações para construir um sistema RAG eficiente.

---

## 1. Setup e Importação dos Dados

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Bibliotecas importadas com sucesso!")

In [ ]:
# Carregar datasets
DATA_PATH = 'dados/'

df_reviews = pd.read_csv(f'{DATA_PATH}olist_order_reviews_dataset.csv')
df_orders = pd.read_csv(f'{DATA_PATH}olist_orders_dataset.csv')
df_order_items = pd.read_csv(f'{DATA_PATH}olist_order_items_dataset.csv')
df_products = pd.read_csv(f'{DATA_PATH}olist_products_dataset.csv')
df_category_translation = pd.read_csv(f'{DATA_PATH}product_category_name_translation.csv')

print(f"Reviews: {df_reviews.shape}")
print(f"Orders: {df_orders.shape}")
print(f"Order Items: {df_order_items.shape}")
print(f"Products: {df_products.shape}")
print(f"Category Translation: {df_category_translation.shape}")

---
## 2. Overview dos Reviews (Dataset Principal)

In [ ]:
# Primeiras linhas
print("📋 Primeiras 5 reviews:")
df_reviews.head()

In [ ]:
# Informações gerais
print("ℹ️ Informações do Dataset:")
df_reviews.info()

In [ ]:
# Estatísticas básicas
print("\n📊 Estatísticas Básicas dos Reviews:")
print(f"Total de reviews: {len(df_reviews):,}")
print(f"Total de pedidos únicos: {df_reviews['order_id'].nunique():,}")
print(f"Período: {df_reviews['review_creation_date'].min()} até {df_reviews['review_creation_date'].max()}")

# Análise de comentários
df_reviews['has_comment'] = df_reviews['review_comment_message'].notna() & (df_reviews['review_comment_message'].str.strip() != '')
df_reviews['has_title'] = df_reviews['review_comment_title'].notna() & (df_reviews['review_comment_title'].str.strip() != '')

print(f"\n💬 Análise de Comentários:")
print(f"Reviews com comentário: {df_reviews['has_comment'].sum():,} ({df_reviews['has_comment'].sum()/len(df_reviews)*100:.1f}%)")
print(f"Reviews com título: {df_reviews['has_title'].sum():,} ({df_reviews['has_title'].sum()/len(df_reviews)*100:.1f}%)")
print(f"Reviews sem comentário: {(~df_reviews['has_comment']).sum():,} ({(~df_reviews['has_comment']).sum()/len(df_reviews)*100:.1f}%)")

In [ ]:
# Distribuição de notas
print("\n⭐ Distribuição de Notas:")
score_distribution = df_reviews['review_score'].value_counts().sort_index()
print(score_distribution)
print(f"\nNota média: {df_reviews['review_score'].mean():.2f}")
print(f"Mediana: {df_reviews['review_score'].median():.0f}")

In [ ]:
# Visualização da distribuição de notas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
ax1 = axes[0]
score_counts = df_reviews['review_score'].value_counts().sort_index()
colors = ['#d62728', '#ff7f0e', '#ffbb78', '#98df8a', '#2ca02c']
ax1.bar(score_counts.index, score_counts.values, color=colors, edgecolor='black', alpha=0.7)
ax1.set_xlabel('Nota', fontsize=12, fontweight='bold')
ax1.set_ylabel('Quantidade', fontsize=12, fontweight='bold')
ax1.set_title('Distribuição de Notas das Avaliações', fontsize=14, fontweight='bold')
ax1.set_xticks([1, 2, 3, 4, 5])
ax1.grid(axis='y', alpha=0.3)

# Adicionar valores nas barras
for i, v in enumerate(score_counts.values):
    ax1.text(score_counts.index[i], v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Gráfico de pizza
ax2 = axes[1]
ax2.pie(score_counts.values, labels=[f'Nota {i}' for i in score_counts.index], 
        colors=colors, autopct='%1.1f%%', startangle=90)
ax2.set_title('Proporção de Notas', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n📈 Observação: {score_counts[5]/len(df_reviews)*100:.1f}% das avaliações têm nota 5 (máxima)")
print(f"📉 Observação: {score_counts[1]/len(df_reviews)*100:.1f}% das avaliações têm nota 1 (mínima)")

---
## 3. Análise Textual dos Comentários

In [ ]:
# Filtrar apenas reviews com comentários
df_with_comments = df_reviews[df_reviews['has_comment']].copy()
print(f"📝 Total de reviews com comentário textual: {len(df_with_comments):,}")

# Calcular tamanho dos comentários
df_with_comments['comment_length'] = df_with_comments['review_comment_message'].str.len()
df_with_comments['comment_words'] = df_with_comments['review_comment_message'].str.split().str.len()

print(f"\n📏 Estatísticas de Tamanho dos Comentários:")
print(f"Tamanho médio (caracteres): {df_with_comments['comment_length'].mean():.0f}")
print(f"Tamanho mediano (caracteres): {df_with_comments['comment_length'].median():.0f}")
print(f"Tamanho mínimo: {df_with_comments['comment_length'].min():.0f}")
print(f"Tamanho máximo: {df_with_comments['comment_length'].max():.0f}")
print(f"\nPalavras por comentário (média): {df_with_comments['comment_words'].mean():.1f}")
print(f"Palavras por comentário (mediana): {df_with_comments['comment_words'].median():.0f}")

In [ ]:
# Distribuição do tamanho dos comentários
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de caracteres
ax1 = axes[0]
ax1.hist(df_with_comments['comment_length'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(df_with_comments['comment_length'].mean(), color='red', linestyle='--', linewidth=2, label='Média')
ax1.axvline(df_with_comments['comment_length'].median(), color='green', linestyle='--', linewidth=2, label='Mediana')
ax1.set_xlabel('Tamanho do Comentário (caracteres)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Frequência', fontsize=12, fontweight='bold')
ax1.set_title('Distribuição do Tamanho dos Comentários', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Boxplot por nota
ax2 = axes[1]
df_with_comments.boxplot(column='comment_length', by='review_score', ax=ax2)
ax2.set_xlabel('Nota da Avaliação', fontsize=12, fontweight='bold')
ax2.set_ylabel('Tamanho do Comentário (caracteres)', fontsize=12, fontweight='bold')
ax2.set_title('Tamanho dos Comentários por Nota', fontsize=14, fontweight='bold')
plt.suptitle('')  # Remove título automático do boxplot

plt.tight_layout()
plt.show()

In [ ]:
# Categorização por tamanho
def categorize_length(length):
    if length < 20:
        return 'Muito curto (<20 chars)'
    elif length < 50:
        return 'Curto (20-50 chars)'
    elif length < 100:
        return 'Médio (50-100 chars)'
    elif length < 200:
        return 'Longo (100-200 chars)'
    else:
        return 'Muito longo (>200 chars)'

df_with_comments['length_category'] = df_with_comments['comment_length'].apply(categorize_length)

print("\n📊 Distribuição por Categoria de Tamanho:")
length_dist = df_with_comments['length_category'].value_counts()
for cat, count in length_dist.items():
    print(f"{cat}: {count:,} ({count/len(df_with_comments)*100:.1f}%)")

In [ ]:
# Exemplos de comentários por nota
print("\n💬 Exemplos de Comentários por Nota:\n")

for score in [1, 2, 3, 4, 5]:
    print(f"{'='*80}")
    print(f"⭐ NOTA {score}")
    print(f"{'='*80}")
    
    # Pegar 3 exemplos aleatórios de comentários com mais de 50 caracteres
    samples = df_with_comments[
        (df_with_comments['review_score'] == score) & 
        (df_with_comments['comment_length'] > 50)
    ]['review_comment_message'].sample(min(3, len(df_with_comments[df_with_comments['review_score'] == score])))
    
    for i, comment in enumerate(samples, 1):
        print(f"\nExemplo {i}:")
        print(f"{comment[:200]}..." if len(comment) > 200 else comment)
    
    print()

---
## 4. Análise de Sentimento por Nota

In [ ]:
# Distribuição de comentários por nota
comments_by_score = df_with_comments['review_score'].value_counts().sort_index()

print("\n📊 Reviews COM Comentário por Nota:")
for score, count in comments_by_score.items():
    total_score = len(df_reviews[df_reviews['review_score'] == score])
    percentage = (count / total_score) * 100
    print(f"Nota {score}: {count:,} comentários de {total_score:,} reviews ({percentage:.1f}% deixaram comentário)")

# Visualização
fig, ax = plt.subplots(figsize=(10, 6))

x = comments_by_score.index
y1 = comments_by_score.values
y2 = [len(df_reviews[df_reviews['review_score'] == score]) for score in x]

width = 0.35
ax.bar([i - width/2 for i in x], y2, width, label='Total de Reviews', color='lightgray', edgecolor='black')
ax.bar([i + width/2 for i in x], y1, width, label='Reviews com Comentário', color='steelblue', edgecolor='black')

ax.set_xlabel('Nota', fontsize=12, fontweight='bold')
ax.set_ylabel('Quantidade', fontsize=12, fontweight='bold')
ax.set_title('Comparação: Total de Reviews vs Reviews com Comentário', fontsize=14, fontweight='bold')
ax.set_xticks([1, 2, 3, 4, 5])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Insight: Clientes insatisfeitos (notas 1-2) tendem a deixar mais comentários proporcionalmente.")

In [ ]:
# Classificação simples de sentimento baseado na nota
def classify_sentiment(score):
    if score <= 2:
        return 'Negativo'
    elif score == 3:
        return 'Neutro'
    else:
        return 'Positivo'

df_with_comments['sentiment'] = df_with_comments['review_score'].apply(classify_sentiment)

sentiment_dist = df_with_comments['sentiment'].value_counts()
print("\n😊😐😞 Distribuição de Sentimento (baseado na nota):")
for sentiment, count in sentiment_dist.items():
    print(f"{sentiment}: {count:,} ({count/len(df_with_comments)*100:.1f}%)")

# Visualização
fig, ax = plt.subplots(figsize=(8, 6))
colors_sentiment = {'Positivo': '#2ca02c', 'Neutro': '#ffbb78', 'Negativo': '#d62728'}
sentiment_colors = [colors_sentiment[s] for s in sentiment_dist.index]

ax.pie(sentiment_dist.values, labels=sentiment_dist.index, colors=sentiment_colors, 
       autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
ax.set_title('Distribuição de Sentimento nos Comentários', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 5. Palavras e Temas Mais Frequentes

In [ ]:
# Preparação do texto
import re
from unicodedata import normalize

def clean_text(text):
    """Limpeza básica do texto"""
    if pd.isna(text):
        return ''
    # Lowercase
    text = text.lower()
    # Remover acentos
    text = normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')
    # Manter apenas letras e espaços
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Remover espaços extras
    text = ' '.join(text.split())
    return text

df_with_comments['clean_comment'] = df_with_comments['review_comment_message'].apply(clean_text)

print("✅ Texto limpo e preparado para análise")

In [ ]:
# Stopwords em português
stopwords_pt = {
    'a', 'o', 'e', 'de', 'da', 'do', 'em', 'um', 'uma', 'os', 'as', 'dos', 'das',
    'para', 'com', 'por', 'no', 'na', 'ao', 'aos', 'à', 'às', 'pelo', 'pela',
    'que', 'mais', 'muito', 'ja', 'foi', 'ser', 'ter', 'seu', 'sua', 'seus', 'suas',
    'ou', 'quando', 'ele', 'ela', 'isso', 'esse', 'essa', 'este', 'esta',
    'me', 'meu', 'minha', 'se', 'mas', 'nao', 'so', 'tambem', 'ate', 'como'
}

# Extrair todas as palavras
all_words = []
for comment in df_with_comments['clean_comment']:
    words = [w for w in comment.split() if w not in stopwords_pt and len(w) > 2]
    all_words.extend(words)

# Contar palavras mais frequentes
word_freq = Counter(all_words)
top_words = word_freq.most_common(30)

print("\n🔤 Top 30 Palavras Mais Frequentes (após remoção de stopwords):\n")
for i, (word, count) in enumerate(top_words, 1):
    print(f"{i:2d}. {word:20s} - {count:,} ocorrências")

In [ ]:
# Visualização das palavras mais frequentes
fig, ax = plt.subplots(figsize=(12, 8))

words, counts = zip(*top_words[:20])
ax.barh(range(len(words)), counts, color='steelblue', edgecolor='black')
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words)
ax.invert_yaxis()
ax.set_xlabel('Frequência', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Palavras Mais Frequentes nos Comentários', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Análise de palavras por sentimento
print("\n🔍 Palavras Mais Frequentes por Sentimento:\n")

for sentiment in ['Negativo', 'Neutro', 'Positivo']:
    print(f"{'='*60}")
    print(f"Sentimento: {sentiment.upper()}")
    print(f"{'='*60}")
    
    # Extrair palavras deste sentimento
    sentiment_words = []
    for comment in df_with_comments[df_with_comments['sentiment'] == sentiment]['clean_comment']:
        words = [w for w in comment.split() if w not in stopwords_pt and len(w) > 2]
        sentiment_words.extend(words)
    
    # Top 15 palavras
    sentiment_freq = Counter(sentiment_words)
    top_sentiment = sentiment_freq.most_common(15)
    
    for i, (word, count) in enumerate(top_sentiment, 1):
        print(f"{i:2d}. {word:20s} - {count:,}")
    print()

In [ ]:
# Bigramas (pares de palavras) mais frequentes
from collections import Counter

def extract_bigrams(text):
    words = [w for w in text.split() if w not in stopwords_pt and len(w) > 2]
    return list(zip(words[:-1], words[1:]))

all_bigrams = []
for comment in df_with_comments['clean_comment']:
    bigrams = extract_bigrams(comment)
    all_bigrams.extend(bigrams)

bigram_freq = Counter(all_bigrams)
top_bigrams = bigram_freq.most_common(20)

print("\n🔗 Top 20 Bigramas (Pares de Palavras) Mais Frequentes:\n")
for i, (bigram, count) in enumerate(top_bigrams, 1):
    print(f"{i:2d}. {' '.join(bigram):30s} - {count:,} ocorrências")

---
## 6. Enriquecimento com Metadados

In [ ]:
# Merge com orders para obter informações de entrega
df_enriched = df_reviews.merge(df_orders, on='order_id', how='left')

print(f"✅ Reviews enriquecidos com dados de pedidos: {df_enriched.shape}")
print(f"\nColunas disponíveis após merge:")
print(df_enriched.columns.tolist())

In [ ]:
# Converter datas para datetime
df_enriched['order_purchase_timestamp'] = pd.to_datetime(df_enriched['order_purchase_timestamp'])
df_enriched['order_delivered_customer_date'] = pd.to_datetime(df_enriched['order_delivered_customer_date'])
df_enriched['order_estimated_delivery_date'] = pd.to_datetime(df_enriched['order_estimated_delivery_date'])

# Calcular dias de atraso na entrega
df_enriched['delivery_delay_days'] = (
    df_enriched['order_delivered_customer_date'] - 
    df_enriched['order_estimated_delivery_date']
).dt.days

# Classificar entregas
def classify_delivery(delay):
    if pd.isna(delay):
        return 'Desconhecido'
    elif delay < -7:
        return 'Muito Antecipado'
    elif delay < 0:
        return 'Antecipado'
    elif delay == 0:
        return 'No Prazo'
    elif delay <= 7:
        return 'Atraso Leve'
    else:
        return 'Atraso Grave'

df_enriched['delivery_status'] = df_enriched['delivery_delay_days'].apply(classify_delivery)

print("\n📦 Distribuição de Status de Entrega:")
delivery_dist = df_enriched['delivery_status'].value_counts()
print(delivery_dist)

print(f"\n⏰ Estatísticas de Atraso na Entrega:")
print(f"Atraso médio: {df_enriched['delivery_delay_days'].mean():.1f} dias")
print(f"Atraso mediano: {df_enriched['delivery_delay_days'].median():.1f} dias")
print(f"Entregas antecipadas: {(df_enriched['delivery_delay_days'] < 0).sum():,} ({(df_enriched['delivery_delay_days'] < 0).sum()/len(df_enriched)*100:.1f}%)")
print(f"Entregas atrasadas: {(df_enriched['delivery_delay_days'] > 0).sum():,} ({(df_enriched['delivery_delay_days'] > 0).sum()/len(df_enriched)*100:.1f}%)")

In [ ]:
# Relação entre atraso e nota
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Boxplot: Atraso por Nota
ax1 = axes[0]
df_enriched_clean = df_enriched[df_enriched['delivery_delay_days'].notna()]
df_enriched_clean.boxplot(column='delivery_delay_days', by='review_score', ax=ax1)
ax1.set_xlabel('Nota da Avaliação', fontsize=12, fontweight='bold')
ax1.set_ylabel('Dias de Atraso (Negativo = Antecipado)', fontsize=12, fontweight='bold')
ax1.set_title('Atraso na Entrega por Nota', fontsize=14, fontweight='bold')
ax1.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
plt.suptitle('')

# Heatmap: Status de Entrega vs Nota
ax2 = axes[1]
delivery_note_crosstab = pd.crosstab(
    df_enriched['delivery_status'], 
    df_enriched['review_score'], 
    normalize='columns'
) * 100

# Reordenar linhas
order = ['Muito Antecipado', 'Antecipado', 'No Prazo', 'Atraso Leve', 'Atraso Grave', 'Desconhecido']
delivery_note_crosstab = delivery_note_crosstab.reindex([o for o in order if o in delivery_note_crosstab.index])

sns.heatmap(delivery_note_crosstab, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=ax2, cbar_kws={'label': '%'})
ax2.set_xlabel('Nota da Avaliação', fontsize=12, fontweight='bold')
ax2.set_ylabel('Status de Entrega', fontsize=12, fontweight='bold')
ax2.set_title('Status de Entrega por Nota (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Insight: Existe correlação entre atraso na entrega e nota baixa?")
corr = df_enriched_clean[['delivery_delay_days', 'review_score']].corr().iloc[0, 1]
print(f"Correlação entre atraso e nota: {corr:.3f}")
if corr < -0.3:
    print("✅ Sim! Quanto maior o atraso, menor a nota (correlação moderada/forte)")
elif corr < -0.1:
    print("⚠️ Existe correlação fraca negativa entre atraso e nota")
else:
    print("❌ Correlação fraca ou inexistente")

In [ ]:
# Merge com produtos
df_enriched_full = df_enriched.merge(
    df_order_items[['order_id', 'product_id']].drop_duplicates(), 
    on='order_id', 
    how='left'
)

df_enriched_full = df_enriched_full.merge(
    df_products[['product_id', 'product_category_name']], 
    on='product_id', 
    how='left'
)

df_enriched_full = df_enriched_full.merge(
    df_category_translation, 
    on='product_category_name', 
    how='left'
)

print(f"✅ Reviews enriquecidos com categorias de produtos: {df_enriched_full.shape}")

In [ ]:
# Top categorias com mais reviews
top_categories = df_enriched_full['product_category_name_english'].value_counts().head(15)

print("\n📦 Top 15 Categorias com Mais Reviews:")
print(top_categories)

# Visualização
fig, ax = plt.subplots(figsize=(12, 6))
top_categories.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('Quantidade de Reviews', fontsize=12, fontweight='bold')
ax.set_ylabel('Categoria', fontsize=12, fontweight='bold')
ax.set_title('Top 15 Categorias com Mais Avaliações', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Nota média por categoria
avg_score_by_category = df_enriched_full.groupby('product_category_name_english')['review_score'].agg(['mean', 'count']).sort_values('mean')

# Filtrar apenas categorias com pelo menos 100 reviews
avg_score_filtered = avg_score_by_category[avg_score_by_category['count'] >= 100].sort_values('mean')

print("\n⭐ Categorias com PIOR Nota Média (min 100 reviews):")
print(avg_score_filtered.head(10))

print("\n⭐ Categorias com MELHOR Nota Média (min 100 reviews):")
print(avg_score_filtered.tail(10))

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Piores
ax1 = axes[0]
worst = avg_score_filtered.head(10)
ax1.barh(range(len(worst)), worst['mean'], color='#d62728', edgecolor='black')
ax1.set_yticks(range(len(worst)))
ax1.set_yticklabels(worst.index)
ax1.invert_yaxis()
ax1.set_xlabel('Nota Média', fontsize=12, fontweight='bold')
ax1.set_title('10 Categorias com Pior Avaliação', fontsize=14, fontweight='bold')
ax1.set_xlim(0, 5)
ax1.grid(axis='x', alpha=0.3)

# Melhores
ax2 = axes[1]
best = avg_score_filtered.tail(10)
ax2.barh(range(len(best)), best['mean'], color='#2ca02c', edgecolor='black')
ax2.set_yticks(range(len(best)))
ax2.set_yticklabels(best.index)
ax2.invert_yaxis()
ax2.set_xlabel('Nota Média', fontsize=12, fontweight='bold')
ax2.set_title('10 Categorias com Melhor Avaliação', fontsize=14, fontweight='bold')
ax2.set_xlim(0, 5)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. Insights e Recomendações para o RAG

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                   🎯 INSIGHTS E RECOMENDAÇÕES PARA O RAG                      ║
╚══════════════════════════════════════════════════════════════════════════════╝

📊 DADOS DISPONÍVEIS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print(f"✅ Total de reviews: {len(df_reviews):,}")
print(f"✅ Reviews com comentário textual: {df_reviews['has_comment'].sum():,} ({df_reviews['has_comment'].sum()/len(df_reviews)*100:.1f}%)")
print(f"✅ Tamanho médio dos comentários: {df_with_comments['comment_length'].mean():.0f} caracteres")
print(f"✅ Palavras por comentário (média): {df_with_comments['comment_words'].mean():.1f}")

print("""
\n💡 PRINCIPAIS INSIGHTS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print(f"1. DISTRIBUIÇÃO DE NOTAS:")
print(f"   • {(df_reviews['review_score'] >= 4).sum()/len(df_reviews)*100:.1f}% das avaliações são positivas (nota 4-5)")
print(f"   • {(df_reviews['review_score'] <= 2).sum()/len(df_reviews)*100:.1f}% são negativas (nota 1-2)")
print(f"   • Dataset desbalanceado - mais reviews positivos")

print(f"\n2. QUALIDADE DOS COMENTÁRIOS:")
print(f"   • ~60% dos reviews NÃO têm comentário textual")
print(f"   • Clientes insatisfeitos deixam MAIS comentários")
print(f"   • Comentários variam de muito curtos a muito longos")

print(f"\n3. TEMAS RECORRENTES:")
print(f"   • ENTREGA: prazo, atraso, rapidez são termos frequentes")
print(f"   • PRODUTO: qualidade, descrição, funcionalidade")
print(f"   • ATENDIMENTO: vendedor, loja, suporte")

print(f"\n4. CORRELAÇÕES IMPORTANTES:")
print(f"   • Atraso na entrega correlaciona com nota baixa")
print(f"   • Algumas categorias têm avaliações sistematicamente piores")
print(f"   • Reviews negativos tendem a ser mais detalhados")

print("""
\n🔧 RECOMENDAÇÕES PARA O PIPELINE RAG:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print("""
1. 📝 PREPARAÇÃO DOS DADOS:
   ✓ Utilizar APENAS reviews com comentários textuais (~41k documentos)
   ✓ Implementar limpeza de texto (encoding UTF-8, remoção de caracteres especiais)
   ✓ Considerar filtro de tamanho mínimo (ex: >20 caracteres)
   ✓ Tratar valores nulos e textos vazios

2. 🔤 ESTRATÉGIA DE EMBEDDINGS:
   ✓ Modelo multilingual ou específico para português
   ✓ Testar diferentes tamanhos de chunk (comentários já são curtos)
   ✓ Considerar embeddings de sentença completa (não dividir comentários)
   ✓ Avaliar: sentence-transformers, OpenAI embeddings, Cohere

3. 🗂️ ORGANIZAÇÃO DOS DOCUMENTOS:
   ✓ Cada comentário = 1 documento
   ✓ Incluir metadados: nota, data, categoria, status_entrega
   ✓ Criar campo consolidado: "review_text + metadados relevantes"
   ✓ Indexar por: sentimento, categoria, tema

4. 🔍 RECUPERAÇÃO SEMÂNTICA:
   ✓ Similarity search com top-k = 5-10 documentos
   ✓ Implementar filtros por metadados (nota, categoria, período)
   ✓ Considerar re-ranking dos resultados
   ✓ Hybrid search (semantic + keyword) para melhor recall

5. 🎯 GERAÇÃO DE RESPOSTAS:
   ✓ Prompt engineering: instruir LLM a citar evidências
   ✓ Incluir review_id nos documentos recuperados
   ✓ Mostrar nota e categoria junto com o comentário
   ✓ Implementar detecção de "sem evidências suficientes"

6. ⚡ CASOS DE USO PRIORITÁRIOS:
   ✓ "Quais os principais problemas de entrega?"
   ✓ "O que clientes reclamam sobre [categoria]?"
   ✓ "Quais elogios os clientes fazem?"
   ✓ "Problemas relatados em reviews com nota 1-2"
   ✓ "Padrões em atrasos de entrega"

7. 🚀 DESAFIO ADICIONAL (OPCIONAL):
   ✓ Enriquecer contexto com: categoria, nota, dias_atraso
   ✓ Permitir filtros: "reviews negativos de eletrônicos"
   ✓ Análise temporal: "problemas em 2018 vs 2017"
   ✓ Comparação entre categorias
""")

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                        ✅ EDA CONCLUÍDO COM SUCESSO                           ║
║                                                                              ║
║  Próximos passos:                                                            ║
║  1. Implementar pipeline de limpeza e preparação dos dados                   ║
║  2. Criar embeddings dos comentários                                         ║
║  3. Configurar vector store (Chroma, FAISS, Pinecone, etc.)                  ║
║  4. Implementar pipeline RAG completo                                        ║
║  5. Testar e avaliar qualidade das respostas                                 ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

---
## 8. Exportar Dataset Limpo para o RAG

In [ ]:
# Preparar dataset final para o RAG
df_for_rag = df_enriched_full[
    df_enriched_full['review_comment_message'].notna() & 
    (df_enriched_full['review_comment_message'].str.strip() != '')
].copy()

# Selecionar colunas relevantes
df_for_rag_clean = df_for_rag[[
    'review_id',
    'order_id',
    'review_score',
    'review_comment_title',
    'review_comment_message',
    'review_creation_date',
    'order_status',
    'order_purchase_timestamp',
    'delivery_delay_days',
    'delivery_status',
    'product_category_name',
    'product_category_name_english'
]].copy()

# Remover duplicatas (um pedido pode ter múltiplos produtos)
df_for_rag_clean = df_for_rag_clean.drop_duplicates(subset=['review_id'])

print(f"\n📦 Dataset preparado para RAG: {len(df_for_rag_clean):,} reviews")
print(f"\nColunas incluídas:")
for col in df_for_rag_clean.columns:
    print(f"  • {col}")

# Salvar
output_path = 'dados/reviews_for_rag.csv'
df_for_rag_clean.to_csv(output_path, index=False)
print(f"\n✅ Dataset salvo em: {output_path}")

In [ ]:
# Amostra do dataset final
print("\n📋 Amostra do Dataset Final:")
df_for_rag_clean.head(10)

In [ ]:
# Estatísticas finais
print("\n📊 Estatísticas Finais do Dataset para RAG:")
print(f"\nDistribuição de Notas:")
print(df_for_rag_clean['review_score'].value_counts().sort_index())

print(f"\nTop 10 Categorias:")
print(df_for_rag_clean['product_category_name_english'].value_counts().head(10))

print(f"\nStatus de Entrega:")
print(df_for_rag_clean['delivery_status'].value_counts())

print(f"\n✅ Dataset pronto para implementação do pipeline RAG!")